<a href="https://colab.research.google.com/github/RayenMarzouk6/Simulation-de-Propagation-d-pid-mie-sur-une-Grille/blob/main/CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sat May 16 23:11:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile epidemic_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>

#define N 512
#define T_MAX 200

#define S 0
#define I 1
#define R 2

__global__ void epidemic_step(int *grid, int *new_grid)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= N || y >= N)
        return;

    int idx = x * N + y;

    int state = grid[idx];

    if (state == I)
    {
        new_grid[idx] = R;
    }
    else if (state == S)
    {
        int infected = 0;

        for (int dx = -1; dx <= 1; dx++)
        {
            for (int dy = -1; dy <= 1; dy++)
            {
                if (dx == 0 && dy == 0)
                    continue;

                int nx = x + dx;
                int ny = y + dy;

                if (nx >= 0 && nx < N &&
                    ny >= 0 && ny < N)
                {
                    if (grid[nx * N + ny] == I)
                        infected++;
                }
            }
        }

        if (infected > 0)
            new_grid[idx] = I;
        else
            new_grid[idx] = S;
    }
    else
    {
        new_grid[idx] = R;
    }
}

int main()
{
    int size = N * N * sizeof(int);

    int *h_grid = (int*)malloc(size);

    for (int i = 0; i < N * N; i++)
        h_grid[i] = S;

    int center = N / 2;

    for (int i = center - 2; i <= center + 2; i++)
    {
        for (int j = center - 2; j <= center + 2; j++)
        {
            h_grid[i * N + j] = I;
        }
    }

    int *d_grid;
    int *d_new_grid;

    cudaMalloc(&d_grid, size);
    cudaMalloc(&d_new_grid, size);

    cudaMemcpy(d_grid, h_grid, size, cudaMemcpyHostToDevice);

    dim3 threads(16,16);
    dim3 blocks((N + 15)/16, (N + 15)/16);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    for (int t = 0; t < T_MAX; t++)
    {
        epidemic_step<<<blocks, threads>>>(d_grid, d_new_grid);

        int *tmp = d_grid;
        d_grid = d_new_grid;
        d_new_grid = tmp;
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_grid, d_grid, size, cudaMemcpyDeviceToHost);

    int S_count = 0;
    int I_count = 0;
    int R_count = 0;

    for (int i = 0; i < N * N; i++)
    {
        if (h_grid[i] == S) S_count++;
        else if (h_grid[i] == I) I_count++;
        else R_count++;
    }

    printf("\n========================================\n");
    printf(" CUDA Simulation Finished\n");
    printf("========================================\n");
    printf("Grid Size    : %d x %d\n", N, N);
    printf("Iterations   : %d\n", T_MAX);
    printf("Susceptible  : %d\n", S_count);
    printf("Infected     : %d\n", I_count);
    printf("Recovered    : %d\n", R_count);
    printf("Execution    : %.4f ms\n", ms);
    printf("========================================\n");

    cudaFree(d_grid);
    cudaFree(d_new_grid);
    free(h_grid);

    return 0;
}

Writing epidemic_cuda.cu


In [ ]:
!nvcc epidemic_cuda.cu -o epidemic_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./epidemic_cuda


 CUDA Simulation Finished
Grid Size    : 512 x 512
Iterations   : 200
Susceptible  : 98119
Infected     : 1616
Recovered    : 162409
Execution    : 129.2246 ms


In [ ]:
%%writefile epidemic_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <curand_kernel.h>

#define N 512
#define T_MAX 200

#define BETA 0.3f
#define GAMMA 0.05f

#define S 0
#define I 1
#define R 2

__global__ void init_rand(curandState *states, unsigned long seed)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= N || y >= N)
        return;

    int id = x * N + y;
    curand_init(seed, id, 0, &states[id]);
}

__global__ void epidemic_step(
    int *grid,
    int *new_grid,
    curandState *states)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= N || y >= N)
        return;

    int idx = x * N + y;
    int state = grid[idx];
    curandState localState = states[idx];

    if (state == I)
    {
        float r = curand_uniform(&localState);
        new_grid[idx] = (r < GAMMA) ? R : I;
    }
    else if (state == S)
    {
        int infected = 0;

        for (int dx = -1; dx <= 1; dx++)
        {
            for (int dy = -1; dy <= 1; dy++)
            {
                if (dx == 0 && dy == 0)
                    continue;

                int nx = x + dx;
                int ny = y + dy;

                if (nx < 0) nx = N - 1;
                if (nx >= N) nx = 0;
                if (ny < 0) ny = N - 1;
                if (ny >= N) ny = 0;

                if (grid[nx * N + ny] == I)
                    infected++;
            }
        }

        float prob = 1.0f;
        for (int k = 0; k < infected; k++)
            prob *= (1.0f - BETA);
        prob = 1.0f - prob;

        float r = curand_uniform(&localState);
        new_grid[idx] = (r < prob) ? I : S;
    }
    else
    {
        new_grid[idx] = R;
    }

    states[idx] = localState;
}

int main()
{
    int size = N * N * sizeof(int);
    int *h_grid = (int*)malloc(size);

    for (int i = 0; i < N * N; i++)
        h_grid[i] = S;

    int center = N / 2;
    for (int i = center - 2; i <= center + 2; i++)
        for (int j = center - 2; j <= center + 2; j++)
            h_grid[i * N + j] = I;

    int *d_grid, *d_new_grid;
    curandState *d_states;

    cudaMalloc(&d_grid, size);
    cudaMalloc(&d_new_grid, size);
    cudaMalloc(&d_states, N * N * sizeof(curandState));

    cudaMemcpy(d_grid, h_grid, size, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((N + 15)/16, (N + 15)/16);

    init_rand<<<blocks, threads>>>(d_states, time(NULL));
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    for (int t = 0; t < T_MAX; t++)
    {
        epidemic_step<<<blocks, threads>>>(d_grid, d_new_grid, d_states);
        cudaDeviceSynchronize();

        int *tmp = d_grid;
        d_grid = d_new_grid;
        d_new_grid = tmp;
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_grid, d_grid, size, cudaMemcpyDeviceToHost);

    int S_count = 0, I_count = 0, R_count = 0;
    for (int i = 0; i < N * N; i++)
    {
        if (h_grid[i] == S) S_count++;
        else if (h_grid[i] == I) I_count++;
        else R_count++;
    }

    printf("\n========================================\n");
    printf(" CUDA Simulation Finished\n");
    printf("========================================\n");
    printf("Grid Size    : %d x %d\n", N, N);
    printf("Iterations   : %d\n", T_MAX);
    printf("Susceptible  : %d\n", S_count);
    printf("Infected     : %d\n", I_count);
    printf("Recovered    : %d\n", R_count);
    printf("Execution    : %.4f ms\n", ms);
    printf("========================================\n");

    cudaFree(d_grid);
    cudaFree(d_new_grid);
    cudaFree(d_states);
    free(h_grid);

    return 0;
}

Writing epidemic_cuda.cu


In [ ]:
!nvcc epidemic_cuda.cu -o epidemic -lcurand && ./epidemic

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).

 CUDA Simulation Finished
Grid Size    : 512 x 512
Iterations   : 200
Susceptible  : 155319
Infected     : 19452
Recovered    : 87373
Execution    : 57.7444 ms
